# Prepare matrices for SCENICplus

Plan: Run SCENIC+ on very high resolution clusters of MOFA factors to resemble 'metacells'

In [1]:
###################
## Load packages
###################
suppressPackageStartupMessages({
    library(scran)
    library(scater)
    library(Seurat)
    library(ArchR)
})


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [42]:
###################
## I/O
###################
io = list()
io$basedir = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/'

## Single-cell
io$RNA_sce = file.path(io$basedir,"data/processed/rna/SingleCellExperiment.rds")
io$meta = file.path(io$basedir,"results/atac/archR/qc/sample_metadata_after_qc.txt.gz")

# ArchR vitro
io$archrvitro = file.path("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/processed/atac/archR")
# ArchR vivo
io$archrvivo = file.path(io$basedir,"data/processed/atac/archR")

# output
io$outdir = file.path("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/multiome_atlas/SCENICplus/")
dir.create(io$outdir, recursive=TRUE, showWarnings =FALSE)

In [3]:
meta = fread(io$meta)[pass_rnaQC == T & pass_atacQC == T & doublet_call == F & !sample %in% c('E8.5_CRISPR_T_KO', 'E8.5_CRISPR_T_WT')]
nrow(meta)

[1] 45713

In [4]:
archrvivo = loadArchRProject(io$archrvivo)[meta$cell]
archrvitro = loadArchRProject(io$archrvitro)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [5]:
archrvivo


           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    



class: ArchRProject 
outputDirectory: /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/data/processed/atac/archR 
samples(11): E7.5_rep1 E7.5_rep2 ... E8.5_CRISPR_T_KO E8.5_CRISPR_T_WT
sampleColData names(1): ArrowFiles
cellColData names(35): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 45713
medianTSS(1): 16.355
medianFrags(1): 30489

In [6]:
# Add in vitro peak matrix to vivo ArchR object
vitroPeaks = getPeakSet(archrvitro)

In [7]:
addArchRThreads(24)
archrvivo = addFeatureMatrix(
              input = archrvivo,
              features = vitroPeaks,
              matrixName = "vitroPeakMatrix",
              ceiling = 4,
              binarize = FALSE,
              verbose = TRUE,
              threads = getArchRThreads(),
              parallelParam = NULL,
              force = TRUE,
              logFile = createLogFile("vitroPeakMatrix")
            )

Setting default number of Parallel threads to 24.

ArchR logging to : ArchRLogs/ArchR-vitroPeakMatrix-12d1367c322135-Date-2024-10-28_Time-12-22-08.log
If there is an issue, please report to github with logFile!

2024-10-28 12:22:11 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-vitroPeakMatrix-12d1367c322135-Date-2024-10-28_Time-12-22-08.log



In [8]:
addArchRThreads(5)
atac.sce = getMatrixFromProject(archrvivo, useMatrix = 'vitroPeakMatrix')

Setting default number of Parallel threads to 5.

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-12d1366b71d4b-Date-2024-10-28_Time-12-25-56.log
If there is an issue, please report to github with logFile!

2024-10-28 12:27:05 : Organizing colData, 1.157 mins elapsed.

2024-10-28 12:27:05 : Organizing rowData, 1.165 mins elapsed.

2024-10-28 12:27:05 : Organizing rowRanges, 1.165 mins elapsed.

2024-10-28 12:27:05 : Organizing Assays (1 of 1), 1.165 mins elapsed.

2024-10-28 12:27:43 : Constructing SummarizedExperiment, 1.784 mins elapsed.

2024-10-28 12:28:28 : Finished Matrix Creation, 2.546 mins elapsed.



In [9]:
######
## Create high resolution clusters
######

# Load RNA SingleCellExperiment
rna.sce <- readRDS(io$RNA_sce)
#rna.sce = rna.sce[,match(meta$cell, colnames(rna.sce))]
# Make sure that samples are consistent
cells <- intersect(colnames(rna.sce),colnames(atac.sce))
rna.sce <- rna.sce[,cells]
atac.sce <- atac.sce[,cells]

In [11]:
rna.sce
atac.sce

class: SingleCellExperiment 
dim: 32285 45707 
metadata(0):
assays(1): counts
rownames(32285): Xkr4 Gm1992 ... AC234645.1 AC149090.1
rowData names(0):
colnames(45707): E7.5_rep1#AAACAGCCATCCTGAA-1
  E7.5_rep1#AAACAGCCATGCTATG-1 ... E8.75_rep2#TTTGTTGGTTCACTGT-1
  E8.75_rep2#TTTGTTGGTTGAGCCG-1
colData names(10): barcode sample ... pass_rnaQC sizeFactor
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

class: RangedSummarizedExperiment 
dim: 234908 45707 
metadata(0):
assays(1): vitroPeakMatrix
rownames: NULL
rowData names(1): idx
colnames(45707): E7.5_rep1#AAACAGCCATCCTGAA-1
  E7.5_rep1#AAACAGCCATGCTATG-1 ... E8.75_rep2#TTTGTTGGTTCACTGT-1
  E8.75_rep2#TTTGTTGGTTGAGCCG-1
colData names(35): BlacklistRatio nDiFrags ... ReadsInPeaks FRIP

In [10]:
# Get mofa object
mofa = readRDS(file.path(io$basedir, '/results/rna_atac/mofa/mofa.rds'))
# Extract factors



In [24]:
mofa_factors <- get_factors(mofa, factors = "all")$group1
mofa_factors = mofa_factors[cells, ]


In [31]:
# Create Seurat object
counts = matrix(ncol=length(colnames(rna.sce)), nrow=1)
colnames(counts) = colnames(rna.sce)
rownames(counts) = 'x'
seurat = CreateSeuratObject(counts)                
seurat[["mofa"]] <- CreateDimReducObject(embeddings = mofa_factors, key = "MOFA_", assay = DefaultAssay(seurat))


In [33]:
# high clustering resolution 
library(future)
plan("multisession", workers = 24)
seurat <- FindNeighbors(seurat,reduction = 'mofa', dims = 1:dim(mofa_factors)[2])
seurat <- FindClusters(seurat, resolution = 200)

Computing nearest neighbor graph

Computing SNN

Warning message:
“The following arguments are not used: reduction”
Warning message:
“The following arguments are not used: reduction”


Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 45707
Number of edges: 1556249

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.3097
Number of communities: 1315
Elapsed time: 7 seconds


13 singletons identified. 1302 final clusters.

Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via the L'Ecuyer-CMRG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”


In [34]:
clusters = as.data.table(seurat@meta.data, keep.rownames=T) %>% .[,c('rn', 'seurat_clusters')] %>% setnames(c('cell', 'cluster'))

In [35]:
library(BiocParallel)
BPPARAM = MulticoreParam(24)

In [37]:
atac.sce = as(atac.sce, 'SingleCellExperiment')

In [39]:
# Pseudobulk by clusters
rna.pb = aggregateAcrossCells(rna.sce, id=clusters$cluster, BPPARAM = BPPARAM)
atac.pb = aggregateAcrossCells(atac.sce, use.assay.type = 'vitroPeakMatrix', id=clusters$cluster, BPPARAM = BPPARAM)

In [ ]:
# Add peak name to atac.pb

In [44]:
rownames(atac.pb) = paste0(seqnames(vitroPeaks), ':', start(vitroPeaks), '-', end(vitroPeaks))

In [61]:
bed = data.frame(chr = seqnames(vitroPeaks),
                 start = start(vitroPeaks), 
                 end = end(vitroPeaks))

In [63]:
fwrite(bed, col.names = F, file.path(io$outdir, 'regions.bed'), sep = '\t')

In [73]:
## Save output for SCENIC+
write.table(colnames(rna.pb), file.path(io$outdir, 'cells.txt'))
write.table(rownames(rna.pb), file.path(io$outdir, 'genes.txt'))
write.table(rownames(atac.pb), file.path(io$outdir, 'peaks.txt'), sep = '\t', row.names = F)

# Save RNA as mtx file
writeMM(as(counts(rna.pb), "dgCMatrix"), file=file.path(io$outdir, 'rna.mtx'))
assayNames(atac.pb) = 'counts'
writeMM(as(counts(atac.pb), "dgCMatrix"), file=file.path(io$outdir, 'atac.mtx'))


NULL

In [114]:
# Save cluster metadata
clusters = as.data.table(seurat@meta.data, keep.rownames=T) %>% .[,c('rn', 'seurat_clusters')] %>% setnames(c('cell', 'cluster'))
celltypes_count = meta %>% .[,.(cell, celltype)] %>%
    merge(., clusters, by = 'cell') %>%
    .[, N := .N, by = c('celltype', 'cluster')] %>%
    .[order(-N)] %>%
    unique(by = c('cluster'))

fwrite(celltypes_count[,.(cluster, celltype)], file.path(io$outdir, 'cluster_metadata.txt'))